# 01 · Auditoría de esquema y relaciones — Encuesta de Movilidad Bogotá 2023

**Objetivo.** Verificar que hogares, personas, viajes y zonificación pueden enlazarse de forma reproducible antes de construir la red estocástica.

Este notebook constituye la **Compuerta 2: integridad analítica**. No estima todavía tiempos de primer paso ni produce resultados sustantivos.


In [1]:
%pip -q install pandas geopandas openpyxl pyreadstat pyarrow requests


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import unicodedata
import zipfile

import geopandas as gpd
import pandas as pd
import pyreadstat
import requests

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023"
OUT = ROOT / "outputs/bogota_em2023"
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

print("Workspace:", ROOT)


Workspace: /content/colombia-stochastic-access


## 1. Fuentes oficiales

Se descargan dos paquetes del Observatorio de Movilidad de Bogotá:

1. base de datos procesada de la Encuesta de Movilidad 2023;
2. zonificación EODH con las capas UTAM y ZAT.


In [3]:
SOURCES = {
    "processed_edoh": {
        "url": "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/05_Base%20datos%20procesada%20EODH.zip",
        "description": "Base procesada EODH 2023",
    },
    "zoning_edoh": {
        "url": "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/03_Zonificacion%20EODH%20%281%29.zip",
        "description": "Zonificación UTAM y ZAT",
    },
}
pd.DataFrame(SOURCES).T


,url,description
processed_edoh,https://observatorio.movilidadbogota.gov.co/si...,Base procesada EODH 2023
zoning_edoh,https://observatorio.movilidadbogota.gov.co/si...,Zonificación UTAM y ZAT


In [4]:
def sha256(path, block_size=1024**2):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def download(url, destination, timeout=600):
    destination = Path(destination)
    if destination.exists() and destination.stat().st_size > 0:
        return {
            "url": url,
            "path": str(destination),
            "bytes": destination.stat().st_size,
            "sha256": sha256(destination),
            "retrieved_utc": "cached",
        }

    temporary = destination.with_suffix(destination.suffix + ".part")
    with requests.get(url, stream=True, timeout=timeout) as response:
        response.raise_for_status()
        total = int(response.headers.get("content-length", 0))
        completed = 0
        with temporary.open("wb") as stream:
            for block in response.iter_content(1024**2):
                if block:
                    stream.write(block)
                    completed += len(block)
                    if total:
                        print(f"\r{destination.name}: {100 * completed / total:5.1f}%", end="")
    temporary.replace(destination)
    print()
    return {
        "url": url,
        "path": str(destination),
        "bytes": destination.stat().st_size,
        "sha256": sha256(destination),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    }


manifest = []
archives = {}
for source_id, source in SOURCES.items():
    archive = RAW / f"{source_id}.zip"
    manifest.append(download(source["url"], archive))
    archives[source_id] = archive

manifest_df = pd.DataFrame(manifest)
manifest_df.to_csv(OUT / "download_manifest.csv", index=False)
manifest_df


,url,path,bytes,sha256,retrieved_utc
0,https://observatorio.movilidadbogota.gov.co/si...,/content/colombia-stochastic-access/data/raw/b...,88826883,2509410f39713de4768872952e75e95809a99d7edf5b65...,cached
1,https://observatorio.movilidadbogota.gov.co/si...,/content/colombia-stochastic-access/data/raw/b...,12225582,405ef62643663e656f58b0595a90a873029809e0aab430...,cached


## 2. Extracción segura e inventario de archivos

También se extraen archivos ZIP anidados, si el paquete oficial los contiene.


In [5]:
def safe_extract(archive, destination):
    archive = Path(archive)
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()

    with zipfile.ZipFile(archive) as bundle:
        for member in bundle.infolist():
            target = (destination / member.filename).resolve()
            if root not in target.parents and target != root:
                raise ValueError(f"Ruta insegura en {archive.name}: {member.filename}")
        bundle.extractall(destination)


def extract_recursively(initial_archives, destination):
    queue = list(initial_archives)
    processed = set()
    while queue:
        archive = Path(queue.pop(0))
        signature = str(archive.resolve())
        if signature in processed:
            continue
        target = destination / re.sub(r"[^A-Za-z0-9_-]+", "_", archive.stem)
        safe_extract(archive, target)
        processed.add(signature)
        queue.extend(
            nested for nested in target.rglob("*.zip")
            if str(nested.resolve()) not in processed
        )
    return processed


extracted_archives = extract_recursively(archives.values(), INTERIM)
all_files = sorted(path for path in INTERIM.rglob("*") if path.is_file())
file_inventory = pd.DataFrame([
    {
        "file": str(path.relative_to(INTERIM)),
        "suffix": path.suffix.lower(),
        "bytes": path.stat().st_size,
    }
    for path in all_files
]).sort_values(["suffix", "file"])

file_inventory.to_csv(OUT / "file_inventory.csv", index=False)
display(file_inventory)
print("Archivos:", len(file_inventory), "· ZIP extraídos:", len(extracted_archives))


,file,suffix,bytes
0,a_Shapefile_UTAM/UTAM2023/UTAM2023.cpg,.cpg,5
7,b_Shapefile_ZAT/ZAT2023/ZAT2023.cpg,.cpg,5
15,processed_edoh/05_Base datos procesada EODH/CS...,.csv,5619008
16,processed_edoh/05_Base datos procesada EODH/CS...,.csv,2944646
17,processed_edoh/05_Base datos procesada EODH/CS...,.csv,35393649
18,processed_edoh/05_Base datos procesada EODH/CS...,.csv,61683330
19,processed_edoh/05_Base datos procesada EODH/CS...,.csv,59436788
1,a_Shapefile_UTAM/UTAM2023/UTAM2023.dbf,.dbf,86420
8,b_Shapefile_ZAT/ZAT2023/ZAT2023.dbf,.dbf,116834
27,zoning_edoh/03_Zonificacion EODH/c. Bitacora c...,.pdf,8234741


Archivos: 28 · ZIP extraídos: 4


## 3. Lectura de tablas y auditoría del esquema

Se leen únicamente muestras pequeñas. El propósito es identificar nombres, tipos y posibles claves sin cargar todavía toda la encuesta en memoria.


In [6]:
TABLE_SUFFIXES = {".csv", ".txt", ".xlsx", ".xls", ".sav", ".dta", ".parquet"}


def normalize(text):
    text = unicodedata.normalize("NFKD", str(text))
    text = "".join(char for char in text if not unicodedata.combining(char))
    return re.sub(r"[^a-z0-9]+", "_", text.lower()).strip("_")


ROLE_TOKENS = {
    "households": ("hogar", "household"),
    "persons": ("persona", "morador", "person"),
    "trips": ("viaje", "trip"),
    "stages": ("etapa", "stage"),
    "vehicles": ("vehiculo", "vehicle"),
}


def infer_role(path):
    name = normalize(path.stem)
    for role, tokens in ROLE_TOKENS.items():
        if any(token in name for token in tokens):
            return role
    return "unclassified"


def read_preview(path, row_limit=500):
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt"}:
        last_error = None
        for encoding in ("utf-8", "latin-1"):
            try:
                return pd.read_csv(
                    path,
                    sep=None,
                    engine="python",
                    nrows=row_limit,
                    encoding=encoding,
                )
            except Exception as exc:
                last_error = exc
        raise last_error
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, nrows=row_limit)
    if suffix == ".sav":
        frame, _ = pyreadstat.read_sav(path, row_limit=row_limit)
        return frame
    if suffix == ".dta":
        return pd.read_stata(path, iterator=True).read(row_limit)
    if suffix == ".parquet":
        return pd.read_parquet(path).head(row_limit)
    raise ValueError(f"Formato no soportado: {suffix}")


schema_rows = []
previews = {}
for path in all_files:
    if path.suffix.lower() not in TABLE_SUFFIXES:
        continue
    relative = str(path.relative_to(INTERIM))
    role = infer_role(path)
    try:
        frame = read_preview(path)
        previews[relative] = frame
        schema_rows.append({
            "file": relative,
            "role": role,
            "readable": True,
            "sample_rows": len(frame),
            "columns": len(frame.columns),
            "column_names": json.dumps(list(map(str, frame.columns)), ensure_ascii=False),
            "error": None,
        })
    except Exception as exc:
        schema_rows.append({
            "file": relative,
            "role": role,
            "readable": False,
            "sample_rows": 0,
            "columns": 0,
            "column_names": "[]",
            "error": f"{type(exc).__name__}: {exc}",
        })

schema_audit = pd.DataFrame(schema_rows)
schema_audit.to_csv(OUT / "schema_audit.csv", index=False)
display(schema_audit[["file", "role", "readable", "sample_rows", "columns", "error"]])


,file,role,readable,sample_rows,columns,error
0,processed_edoh/05_Base datos procesada EODH/CS...,households,True,500,27,None
1,processed_edoh/05_Base datos procesada EODH/CS...,vehicles,True,500,19,None
2,processed_edoh/05_Base datos procesada EODH/CS...,persons,True,500,40,None
3,processed_edoh/05_Base datos procesada EODH/CS...,trips,True,500,47,None
4,processed_edoh/05_Base datos procesada EODH/CS...,stages,True,500,38,None
5,processed_edoh/05_Base datos procesada EODH/XL...,households,True,500,27,None
6,processed_edoh/05_Base datos procesada EODH/XL...,vehicles,True,500,18,None
7,processed_edoh/05_Base datos procesada EODH/XL...,persons,True,500,40,None
8,processed_edoh/05_Base datos procesada EODH/XL...,trips,True,500,47,None
9,processed_edoh/05_Base datos procesada EODH/XL...,stages,True,500,38,None


## 4. Detección de claves y variables científicas

La detección es deliberadamente transparente: genera candidatos que deben ser confirmados con el diccionario oficial.


In [ ]:
COLUMN_PATTERNS = {
    "household_id": ("id_hogar", "idhogar", "hogar_id", "id_hog", "idhog", "hog_id"),
    "person_id": ("id_persona", "idpersona", "persona_id", "id_per", "idper", "per_id"),
    "trip_id": ("id_viaje", "idviaje", "viaje_id", "id_via", "idvia", "via_id"),
    "origin_zone": ("origen", "zona_o", "zat_o", "utam_o"),
    "destination_zone": ("destino", "zona_d", "zat_d", "utam_d"),
    "residence_zone": ("zona_res", "zat_res", "utam_res"),
    "weight": ("factor", "expansion", "ponderacion", "peso", "f_exp", "fexp", "fe_hg", "fe_per", "fe_via"),
    "mode": ("modo", "medio_transporte"),
    "purpose": ("motivo", "proposito"),
    "duration": ("duracion", "tiempo_viaje", "minutos"),
    "stratum": ("estrato",),
    "sex_gender": ("sexo", "genero"),
}


def classify_column(column):
    normalized = normalize(column)
    matches = []
    for concept, patterns in COLUMN_PATTERNS.items():
        if any(pattern in normalized for pattern in patterns):
            matches.append(concept)
    return matches


candidate_rows = []
column_rows = []
role_columns = {}
for relative, frame in previews.items():
    role = infer_role(Path(relative))
    role_columns.setdefault(role, set()).update(normalize(col) for col in frame.columns)
    for column in frame.columns:
        column_rows.append({
            "file": relative,
            "role": role,
            "column": str(column),
            "normalized": normalize(column),
            "dtype_sample": str(frame[column].dtype),
        })
        concepts = classify_column(column)
        if concepts:
            candidate_rows.append({
                "file": relative,
                "role": role,
                "column": str(column),
                "normalized": normalize(column),
                "concepts": ", ".join(concepts),
                "dtype_sample": str(frame[column].dtype),
                "missing_sample_pct": round(100 * frame[column].isna().mean(), 2),
                "unique_sample": int(frame[column].nunique(dropna=True)),
            })

column_catalog = pd.DataFrame(column_rows)
key_candidates = pd.DataFrame(candidate_rows)
column_catalog.to_csv(OUT / "column_catalog.csv", index=False)
key_candidates.to_csv(OUT / "key_candidates.csv", index=False)
display(key_candidates)
print("\nCATÁLOGO COMPLETO DE COLUMNAS NORMALIZADAS")
for role in sorted(role_columns):
    print(f"\n{role}:")
    print(sorted(role_columns[role]))


## 5. Zonificación geográfica


In [8]:
spatial_files = [
    path for path in all_files
    if path.suffix.lower() in {".shp", ".geojson", ".gpkg"}
]
spatial_rows = []
for path in spatial_files:
    try:
        layer = gpd.read_file(path)
        spatial_rows.append({
            "file": str(path.relative_to(INTERIM)),
            "readable": True,
            "features": len(layer),
            "crs": str(layer.crs),
            "geometry_types": ", ".join(sorted(layer.geometry.geom_type.dropna().unique())),
            "columns": json.dumps(list(map(str, layer.columns)), ensure_ascii=False),
            "error": None,
        })
    except Exception as exc:
        spatial_rows.append({
            "file": str(path.relative_to(INTERIM)),
            "readable": False,
            "features": 0,
            "crs": None,
            "geometry_types": None,
            "columns": "[]",
            "error": f"{type(exc).__name__}: {exc}",
        })

spatial_audit = pd.DataFrame(spatial_rows)
spatial_audit.to_csv(OUT / "spatial_audit.csv", index=False)
display(spatial_audit)


,file,readable,features,crs,geometry_types,columns,error
0,a_Shapefile_UTAM/UTAM2023/UTAM2023.shp,True,142,EPSG:4686,"MultiPolygon, Polygon","[""MUNCodigo"", ""MUNNombre"", ""LOCNombre"", ""UTAM""...",None
1,b_Shapefile_ZAT/ZAT2023/ZAT2023.shp,True,1215,EPSG:3116,"MultiPolygon, Polygon","[""OBJECTID"", ""MUNCod"", ""NOMMun"", ""ZAT"", ""UTAM""...",None


## 6. Compuerta 2: integridad analítica

La compuerta no exige todavía que las relaciones sean uno-a-uno. Exige evidencia mínima de que existen tablas legibles, claves de enlace, origen/destino, ponderadores y geometrías.


In [ ]:
readable_schema = schema_audit[schema_audit.readable] if not schema_audit.empty else schema_audit
available_roles = set(readable_schema.role) if not readable_schema.empty else set()
candidate_concepts = set()
if not key_candidates.empty:
    for value in key_candidates.concepts:
        candidate_concepts.update(item.strip() for item in value.split(","))

household_cols = role_columns.get("households", set())
person_cols = role_columns.get("persons", set())
trip_cols = role_columns.get("trips", set())

household_person_shared = sorted(
    column for column in household_cols & person_cols
    if any(token in column for token in ("hogar", "id_hog", "idhog", "hog_id"))
)
person_trip_shared = sorted(
    column for column in person_cols & trip_cols
    if any(token in column for token in ("persona", "person", "id_per", "idper", "per_id"))
)

gate2 = {
    "official_archives_downloaded": len(manifest_df) == 2 and manifest_df.bytes.gt(0).all(),
    "required_tables_readable": {"households", "persons", "trips"}.issubset(available_roles),
    "household_person_key_candidate": bool(household_person_shared) or "household_id" in candidate_concepts,
    "person_trip_key_candidate": bool(person_trip_shared) or "person_id" in candidate_concepts,
    "trip_origin_destination_detected": {"origin_zone", "destination_zone"}.issubset(candidate_concepts),
    "expansion_weight_detected": "weight" in candidate_concepts,
    "zoning_layer_readable": (
        not spatial_audit.empty
        and spatial_audit.readable.any()
        and spatial_audit.features.max() > 0
    ),
}
# Pandas/NumPy return numpy.bool_ in .all() and .any(); JSON needs Python bool.
gate2 = {key: bool(value) for key, value in gate2.items()}
gate2["gate_2_passed"] = bool(all(gate2.values()))

join_report = {
    "household_person_shared": household_person_shared,
    "person_trip_shared": person_trip_shared,
    "available_roles": sorted(available_roles),
    "candidate_concepts": sorted(candidate_concepts),
}
(OUT / "join_candidate_report.json").write_text(
    json.dumps(join_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
(OUT / "gate_2.json").write_text(
    json.dumps(gate2, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(pd.Series(gate2, name="result"))
print(json.dumps(join_report, ensure_ascii=False, indent=2))


## Interpretación del resultado

- Si **gate_2_passed=True**, el siguiente notebook construirá las relaciones completas, comprobará cardinalidades y generará una matriz origen–destino ponderada.
- Si alguna condición es falsa, se revisan los archivos de auditoría para ajustar únicamente el mapeo de nombres.
- Los resultados pequeños quedan en la carpeta outputs/bogota_em2023/.

**No guarde los ZIP ni las bases extraídas en GitHub.** Solo deben versionarse el código, los metadatos y los resultados pequeños.
